# Facial Emotion Recognition — ANN + MediaPipe
### Innomatics Research Labs — Image Data Based ANN Capstone Project

This notebook walks through Steps 1–12 of the capstone:
Problem Statement → Dataset Collection/Import → EDA → Preprocessing →
Feature Extraction (MediaPipe blendshapes) → Input/Output Separation →
Train/Test Split → Scaling → Model Building → Hyperparameter Tuning (Optuna)
→ Evaluation → Model Saving.


## Step 1 : Problem Statement Selection

- **Project Title:** Facial Emotion Recognition using ANN + MediaPipe
- **Problem Statement:** Automatically classify a person's facial emotion from an image.
- **Business Objective:** Power emotion-aware applications (engagement tracking, sentiment kiosks, driver monitoring).
- **Expected Output:** Predicted emotion label with a confidence score.
- **Target Variable:** `emotion` — 7 classes: angry, disgust, fear, happy, neutral, sad, surprise.
- **Input Features:** 52 MediaPipe face-blendshape scores extracted per image.


## Step 2 : Dataset Collection

We use a **folder-per-class** FER-style dataset (e.g. Kaggle's FER2013 folder version):

```
dataset/train/<emotion>/*.jpg
dataset/test/<emotion>/*.jpg
```

Download it from Kaggle and place it under `dataset/` before running this notebook.


In [ ]:
# Step 3 : Dataset Import — imports
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import optuna
import joblib

import sys
sys.path.append('..')
from preprocessing import (build_face_landmarker, resize_image, to_rgb,
                            extract_blendshape_features, preprocess_and_extract, BLENDSHAPE_DIM)

DATASET_DIR = "../dataset"
sns.set_style("darkgrid")


## Step 3 : Dataset Import

In [ ]:
def load_dataset_index(dataset_dir, split="train"):
    split_dir = os.path.join(dataset_dir, split)
    rows = []
    for emotion in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, emotion)
        if not os.path.isdir(class_dir):
            continue
        for img_path in glob.glob(os.path.join(class_dir, "*")):
            rows.append({"filepath": img_path, "emotion": emotion})
    return pd.DataFrame(rows)

train_df = load_dataset_index(DATASET_DIR, "train")
test_df = load_dataset_index(DATASET_DIR, "test")
full_df = pd.concat([train_df, test_df], ignore_index=True)

print("Shape:", full_df.shape)
print("Columns:", list(full_df.columns))
print("Dtypes:\n", full_df.dtypes)
print("Missing values:\n", full_df.isnull().sum())
print("Duplicate rows:", full_df.duplicated().sum())
print("Class distribution:\n", full_df["emotion"].value_counts())


## Step 4 : Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=full_df, x="emotion", order=full_df["emotion"].value_counts().index)
plt.title("Class Distribution — Emotion Counts")
plt.xticks(rotation=45)
plt.show()


In [ ]:
dist = full_df["emotion"].value_counts(normalize=True) * 100
plt.figure(figsize=(6,6))
plt.pie(dist.values, labels=dist.index, autopct="%1.1f%%")
plt.title("Emotion Class Proportion")
plt.show()


In [ ]:
# Visualize a few sample images per class
fig, axes = plt.subplots(1, 7, figsize=(18,3))
for ax, emotion in zip(axes, sorted(full_df["emotion"].unique())):
    sample_path = full_df[full_df["emotion"] == emotion]["filepath"].iloc[0]
    img = cv2.imread(sample_path)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(emotion)
    ax.axis("off")
plt.tight_layout()
plt.show()


**Observations / Insights / Recommendations**

- The dataset is imbalanced across emotion classes (`disgust` is typically the rarest).
- Images are low-resolution (e.g. 48×48 for FER2013) and grayscale — they need
  upscaling + RGB conversion before MediaPipe can reliably detect a face.
- Recommendation: upscale to at least 224×224, and consider class-weighting or
  stratified sampling to counter imbalance during training.


## Step 5 : Data Preprocessing

For image projects: resize, normalize, and (optionally) augment. See
`preprocessing.py` for `resize_image`, `normalize_image`, `to_rgb`, and
`augment_image`.


In [ ]:
IMAGE_SIZE = (224, 224)

sample_img = cv2.imread(full_df["filepath"].iloc[0])
resized = resize_image(sample_img, IMAGE_SIZE)
rgb = to_rgb(resized)

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(rgb); axes[1].set_title("Resized + RGB"); axes[1].axis("off")
plt.show()


## Step 6 : Feature Extraction — MediaPipe Blendshapes

Instead of flattening pixels or using HOG/CNN features, we extract MediaPipe's
**52 face-blendshape scores** per image — these directly represent facial
muscle activation (e.g. `mouthSmileLeft`, `browDownRight`, `jawOpen`) and are
a very strong, compact feature set for expression classification.


In [ ]:
landmarker = build_face_landmarker(model_path="../models/face_landmarker.task", running_mode="IMAGE")

features, labels = [], []
dropped = 0

for i, row in full_df.iterrows():
    img = cv2.imread(row["filepath"])
    if img is None:
        dropped += 1
        continue
    img_resized = resize_image(img, IMAGE_SIZE)
    img_rgb = to_rgb(img_resized)
    vec = extract_blendshape_features(img_rgb, landmarker)
    if vec is None:
        dropped += 1
        continue
    features.append(vec)
    labels.append(row["emotion"])

    if i % 500 == 0:
        print(f"Processed {i}/{len(full_df)}")

print(f"Usable samples: {len(features)}, dropped (no face detected): {dropped}")

X = np.vstack(features)
y = np.array(labels)
np.savez_compressed("../dataset/features_cache.npz", X=X, y=y)


## Step 7 : Input / Output Separation

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("X shape:", X.shape)
print("y shape:", y_encoded.shape)
print("Classes:", list(label_encoder.classes_))


## Step 8 : Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print("Train:", X_train.shape, "Test:", X_test.shape)


## Step 8a : Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 9 : Model Building

Train a few baseline ANN architectures and compare validation accuracy.


In [ ]:
def build_ann(input_dim, num_classes, hidden_layers=(128,64), dropout=0.3,
              learning_rate=1e-3, activation="relu"):
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(num_classes, activation="softmax"))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

num_classes = len(label_encoder.classes_)
architectures = {
    "small_ann":  dict(hidden_layers=(64,32),      dropout=0.2, learning_rate=1e-3),
    "medium_ann": dict(hidden_layers=(128,64),      dropout=0.3, learning_rate=1e-3),
    "deep_ann":   dict(hidden_layers=(256,128,64),  dropout=0.4, learning_rate=5e-4),
}

baseline_results = {}
early_stop = callbacks.EarlyStopping(patience=6, restore_best_weights=True)

for name, params in architectures.items():
    model = build_ann(X_train_scaled.shape[1], num_classes, **params)
    history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                         epochs=30, batch_size=32, verbose=0, callbacks=[early_stop])
    baseline_results[name] = max(history.history["val_accuracy"])
    print(name, "val_accuracy:", baseline_results[name])

baseline_results


## Step 10 : Hyperparameter Tuning (Optuna)

In [ ]:
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layers = tuple(trial.suggest_categorical(f"units_l{i}", [32,64,128,256]) for i in range(n_layers))
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "elu", "tanh"])

    model = build_ann(X_train_scaled.shape[1], num_classes, hidden_layers, dropout, learning_rate, activation)
    early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                         epochs=40, batch_size=32, verbose=0, callbacks=[early_stop])
    return max(history.history["val_accuracy"])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25)

print("Best value:", study.best_value)
print("Best params:", study.best_params)


## Step 11 : Model Evaluation

In [ ]:
best_params = study.best_params
n_layers = best_params["n_layers"]
hidden_layers = tuple(best_params[f"units_l{i}"] for i in range(n_layers))

final_model = build_ann(X_train_scaled.shape[1], num_classes, hidden_layers,
                         best_params["dropout"], best_params["learning_rate"], best_params["activation"])
early_stop = callbacks.EarlyStopping(patience=8, restore_best_weights=True)
final_model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                 epochs=40, batch_size=32, callbacks=[early_stop], verbose=1)


In [ ]:
y_pred_probs = final_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
auc = roc_auc_score(y_test, y_pred_probs, multi_class="ovr", average="weighted")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()


## Step 12 : Model Saving

In [ ]:
os.makedirs("../models", exist_ok=True)
final_model.save("../models/emotion_ann_model.keras")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(label_encoder, "../models/label_encoder.pkl")
print("Saved model, scaler, and label encoder to ../models/")


## Challenges Faced
- Low-resolution source images reduce MediaPipe's face-detection success rate — mitigated by upscaling to 224×224.
- Class imbalance across the 7 emotions (notably `disgust`).

## Future Scope
- Add class-weighting / SMOTE for imbalance.
- Extend to real-time video-stream inference using MediaPipe's `LIVE_STREAM` running mode.
- Support multi-face detection in a single frame.
